# MedBot - Qwen3-4B fine-tuning (Unsloth + QLoRA), completely free
Runtime > Change runtime type > **T4 GPU**. Run the cells in order (about 1-1.5 hours in total).

**If Colab disconnects:** reopen, set T4 again and press *Run all*. Everything is saved on Google Drive (`MyDrive/medbot` folder): dataset generation and training resume where they stopped.

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

## 1. Mount Google Drive and upload chunks.jsonl (first run only; from your PC: data/chunks.jsonl)

In [ ]:
import shutil
import os
from google.colab import drive, files
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/medbot"
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f"{WORK}/chunks.jsonl"):
    files.upload()
    shutil.move("chunks.jsonl", f"{WORK}/chunks.jsonl")
print(os.listdir(WORK))

## 2. Modello + LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B", max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=0)

## 3. Generate the dataset for free with Qwen3 (self-instruct on the chunks, ~30-45 min)
Change N for fewer or more examples.

In [ ]:
import json, random, hashlib
random.seed(0)
N = 1200
SYSTEM = (
 "You are MedBot, an encyclopedic medical reference assistant.\n"
 "Answer ONLY using the numbered context passages. Cite them like [1], [2].\n"
 "If the context does not contain the answer, say you don't have enough information.\n"
 "Write in clear English. You provide general educational information, not personal medical advice: "
 "for questions about a specific person's symptoms, diagnosis or treatment, give general background "
 "and tell the user to consult a qualified healthcare professional; for emergencies, tell them to call emergency services.")
GEN = ("From the passage below, write ONE realistic question a user might ask a medical encyclopedia, answerable from the passage, "
       "plus an answer in 2-6 sentences using ONLY the passage. End the answer with the citation [1]. "
       'Return ONLY JSON: {"question": "...", "answer": "..."}\n\nPassage:\n')
UNANS = ("Write ONE medical question on a related but DIFFERENT topic that the passage below does NOT answer. "
         'Return ONLY JSON: {"question": "..."}\n\nPassage:\n')
NOINFO = "The provided context does not contain enough information to answer this question."
PERSONAL = ["I have had chest pain since this morning, what disease do I have?", "Can you diagnose the rash on my arm?",
  "What dose of antibiotics should I take for my sore throat?", "My child has a fever of 40C, what should I do?",
  "Should I stop taking my blood pressure medication?", "I think I'm having a stroke, my face is drooping."]
PERSONAL_ANS = ("I can't diagnose or recommend treatment for an individual, and this is general educational information only. "
  "Please consult a qualified healthcare professional about your situation. If symptoms are severe or sudden (for example chest pain, "
  "trouble breathing, signs of stroke, or very high fever in a child), call your local emergency number immediately.")

chunks = [json.loads(l) for l in open(f"{WORK}/chunks.jsonl", encoding="utf-8")]
random.shuffle(chunks); chunks = chunks[:N]
FastLanguageModel.for_inference(model); tokenizer.padding_side = "left"

def gen(prompts, max_new=300):
    texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
             add_generation_prompt=True, enable_thinking=False) for p in prompts]
    enc = tokenizer(texts, return_tensors="pt", padding=True).to("cuda")
    out = model.generate(**enc, max_new_tokens=max_new, temperature=0.3, do_sample=True)
    return tokenizer.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)

def parse(t):
    try: return json.loads(t[t.index("{"): t.rindex("}") + 1])
    except Exception: return None

def row(ctx, q, a):
    return {"messages": [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Context:\n[1] {ctx}\n\nQuestion: {q}"},
            {"role": "assistant", "content": a}]}

GEN_F, PROG_F = f"{WORK}/gen_rows.jsonl", f"{WORK}/gen_progress.txt"
start = int(open(PROG_F).read()) if os.path.exists(PROG_F) else 0   # resume where it stopped
if start: print("Resuming generation from", start)
B = 16
for i in range(start, len(chunks), B):
    batch = chunks[i:i+B]
    rows = []
    kinds = [random.random() < 0.12 for _ in batch]   # True = question not answerable from the context
    outs = gen([(UNANS if k else GEN) + c["text"] for k, c in zip(kinds, batch)])
    for k, c, o in zip(kinds, batch, outs):
        j = parse(o)
        if not j or "question" not in j: continue
        if k: rows.append((c["title"], row(c["text"], j["question"], NOINFO)))
        elif j.get("answer") and "[1]" in j["answer"]: rows.append((c["title"], row(c["text"], j["question"], j["answer"])))
    with open(GEN_F, "a", encoding="utf-8") as f:
        for t, r in rows: f.write(json.dumps({"title": t, "row": r}, ensure_ascii=False) + "\n")
    open(PROG_F, "w").write(str(i + B))
    print(i + B, "/", len(chunks))
rows = [(d["title"], d["row"]) for d in map(json.loads, open(GEN_F, encoding="utf-8"))]

val_titles = {t for t, _ in rows if int(hashlib.md5(t.encode()).hexdigest(), 16) % 10 == 0}
train = [r for t, r in rows if t not in val_titles] + [row("(no relevant context)", q, PERSONAL_ANS) for q in PERSONAL * 6]
val = [r for t, r in rows if t in val_titles]
random.shuffle(train)
for n, d in (("train", train), ("val", val)):
    with open(f"{WORK}/{n}.jsonl", "w", encoding="utf-8") as f:
        for r in d: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(len(train), "train,", len(val), "val")
FastLanguageModel.for_training(model)

## 3b. Manually inspect a few examples (dataset quality!)

In [ ]:
for r in random.sample(train, 3):
    print(r["messages"][1]["content"][-400:], "\n->", r["messages"][2]["content"], "\n" + "-"*60)

## 4. Chat-format dataset + training (assistant replies only)

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
ds = load_dataset("json", data_files={"train": f"{WORK}/train.jsonl", "val": f"{WORK}/val.jsonl"})
def fmt(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False)}
ds = ds.map(fmt)
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds["train"], eval_dataset=ds["val"],
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=2048,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=2, learning_rate=2e-4, warmup_ratio=0.05,
        lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.01,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10, eval_strategy="epoch", save_strategy="steps", save_steps=50, save_total_limit=2,
        output_dir=f"{WORK}/out", report_to="none", seed=0))
trainer = train_on_responses_only(
    trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
import glob
resume = bool(glob.glob(f"{WORK}/out/checkpoint-*"))
if resume: print("Resuming training from the latest checkpoint on Drive")
trainer.train(resume_from_checkpoint=resume)
model.save_pretrained(f"{WORK}/lora"); tokenizer.save_pretrained(f"{WORK}/lora")   # adapter saved to Drive

## 5. Quick test

In [ ]:
FastLanguageModel.for_inference(model)
ctx = "[1] Type 2 diabetes is a chronic condition in which the body does not use insulin properly, causing high blood sugar."
msgs = [{"role":"system","content":SYSTEM},{"role":"user","content":f"Context:\n{ctx}\n\nQuestion: What is type 2 diabetes?"}]
inp = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", enable_thinking=False).to("cuda")
out = model.generate(input_ids=inp, max_new_tokens=200, temperature=0.2, do_sample=True)
print(tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True))

## 6. Export GGUF for Ollama and download it

In [ ]:
model.save_pretrained_gguf(f"{WORK}/medbot_gguf", tokenizer, quantization_method="q4_k_m")
import glob
f = glob.glob(f"{WORK}/medbot_gguf*/*.gguf")[0]; print("GGUF on Drive:", f)
from google.colab import files
files.download(f)   # if it fails, download it from Drive